# Experiment 4: Interactive Dashboard with Filters and Slicers

**Aim:** Create an interactive retail dashboard with Region, Category and date filters. Uses ipywidgets inside Jupyter.

## Import libraries and load the supplied CSV files
The five dimension/fact tables are joined using their ID/date keys. Sales, cost and profit are calculated from the supplied unit price, unit cost, units sold and discount.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

DATA_DIR = Path("datasets")

def load_data():
    required_files = [
        "Dim_Customer.csv", "Dim_Date.csv", "Dim_Geography.csv",
        "Dim_Product.csv", "Fact_Orders.csv"
    ]
    missing = [f for f in required_files if not (DATA_DIR / f).exists()]
    if missing:
        raise FileNotFoundError(
            "Missing CSV file(s): " + ", ".join(missing) +
            ". Put all five CSV files in the datasets folder beside this notebook."
        )
    customer = pd.read_csv(DATA_DIR / "Dim_Customer.csv")
    dates = pd.read_csv(DATA_DIR / "Dim_Date.csv")
    geography = pd.read_csv(DATA_DIR / "Dim_Geography.csv")
    product = pd.read_csv(DATA_DIR / "Dim_Product.csv")
    orders = pd.read_csv(DATA_DIR / "Fact_Orders.csv")

    # Join the fact table with dimension tables using their matching keys.
    data = orders.merge(customer, on="Customer_ID", how="left")
    data = data.merge(product, on="Product_ID", how="left")
    data = data.merge(geography, on="Geo_ID", how="left")
    data = data.merge(dates, left_on="Order_Date", right_on="Date", how="left")

    data["Order_Date"] = pd.to_datetime(data["Order_Date"], errors="coerce")
    data["Ship_Date"] = pd.to_datetime(data["Ship_Date"], errors="coerce")
    data["Sales"] = data["Units_Sold"] * data["Unit_Price"] * (1 - data["Discount_Pct"])
    data["Cost"] = data["Units_Sold"] * data["Unit_Cost"]
    data["Profit"] = data["Sales"] - data["Cost"]
    data["Profit_Margin"] = np.where(data["Sales"] != 0,
                                     data["Profit"] / data["Sales"] * 100, 0)
    data["Order_Month"] = data["Order_Date"].dt.to_period("M").astype(str)
    return data

## Python code and visualization

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

df = load_data()

region_filter = widgets.SelectMultiple(
    options=sorted(df["Region"].dropna().unique()),
    value=tuple(sorted(df["Region"].dropna().unique())),
    description="Region", rows=6
)
category_filter = widgets.SelectMultiple(
    options=sorted(df["Category"].dropna().unique()),
    value=tuple(sorted(df["Category"].dropna().unique())),
    description="Category", rows=5
)
start_date = widgets.DatePicker(description="Start date", value=df["Order_Date"].min().date())
end_date = widgets.DatePicker(description="End date", value=df["Order_Date"].max().date())
output = widgets.Output()

def update_dashboard(change=None):
    with output:
        clear_output(wait=True)
        start = pd.Timestamp(start_date.value) if start_date.value else df["Order_Date"].min()
        end = pd.Timestamp(end_date.value) if end_date.value else df["Order_Date"].max()
        filtered = df[
            df["Region"].isin(region_filter.value) &
            df["Category"].isin(category_filter.value) &
            df["Order_Date"].between(start, end)
        ]
        print("Total Sales (₹):", round(filtered["Sales"].sum(), 2))
        print("Total Profit (₹):", round(filtered["Profit"].sum(), 2))
        print("Number of Orders:", filtered["Order_ID"].nunique())
        if not filtered.empty:
            display(filtered.groupby("Category")["Sales"].sum().to_frame())
            fig = px.bar(filtered.groupby("Category", as_index=False)["Sales"].sum(),
                         x="Category", y="Sales", title="Filtered Sales by Category")
            fig.show()
            trend = filtered.groupby("Order_Month", as_index=False)["Sales"].sum()
            px.line(trend, x="Order_Month", y="Sales", markers=True,
                    title="Filtered Monthly Sales").show()
        else:
            print("No records match the selected filters.")

for control in [region_filter, category_filter, start_date, end_date]:
    control.observe(update_dashboard, names="value")

display(widgets.VBox([
    widgets.HBox([region_filter, category_filter]),
    widgets.HBox([start_date, end_date]),
    output
]))
update_dashboard()

## Result
The experiment was performed using the supplied retail dataset. Run all cells to generate the tables, calculations and visualizations.